# Adult Income Prediction



In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://raw.githubusercontent.com/Hanne2202/ml-group10-data/main/adult.csv"
df = pd.read_csv(url)

## 2. Data Pre-processing
Sau bước EDA, nhóm tiến hành tiền xử lý dữ liệu để chuẩn bị cho giai đoạn huấn luyện mô hình.  
Các bước chính gồm:
- chuẩn hóa missing value,
- loại bỏ đặc trưng dư thừa,
- chuẩn bị tập đặc trưng và biến mục tiêu,
- chia train/test,
- xây dựng và so sánh các cấu hình preprocessing cho numerical và categorical features.

### 2.1. Missing value standardization
Kết quả cho thấy dữ liệu thiếu không xuất hiện dưới dạng `NaN` ngay từ đầu mà được mã hóa bằng ký hiệu `?`.  
Sau khi thay thế, các cột có missing value là:
- `workclass`
- `occupation`
- `native-country`

Tỷ lệ thiếu ở các cột này không quá lớn, vì vậy nhóm ưu tiên giữ lại dữ liệu và sẽ xử lý bằng imputation trong pipeline thay vì loại bỏ toàn bộ các dòng.

In [ ]:
# [Preprocessing 2.1] Replace '?' with NaN
df_prep = df.copy()
df_prep.replace('?', np.nan, inplace=True)

print(df_prep.isnull().sum())

age                   0
workclass          2799
fnlwgt                0
education             0
educational-num       0
marital-status        0
occupation         2809
relationship          0
race                  0
gender                0
capital-gain          0
capital-loss          0
hours-per-week        0
native-country      857
income                0
dtype: int64


### 2.2. Remove redundant and non-predictive features
Dựa trên kết quả EDA, nhóm loại bỏ hai cột sau:

- `education`: do trùng lặp thông tin với `educational-num`
- `fnlwgt`: đây là sampling weight của điều tra dân số, không phải đặc trưng mô tả trực tiếp cá nhân và thường không mang nhiều ý nghĩa dự đoán trong bài toán phân loại thu nhập

Việc loại bỏ hai cột này giúp giảm dư thừa thông tin và làm tập đặc trưng phù hợp hơn cho mô hình.

In [ ]:
# [Preprocessing 2.2] Drop redundant / non-predictive features
df_prep.drop(columns=['education', 'fnlwgt'], inplace=True, errors='ignore')

print(df_prep.shape)
print(df_prep.columns.tolist())
print(df_prep.head())

(48842, 13)
['age', 'workclass', 'educational-num', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
   age  workclass  educational-num      marital-status         occupation  \
0   25    Private                7       Never-married  Machine-op-inspct   
1   38    Private                9  Married-civ-spouse    Farming-fishing   
2   28  Local-gov               12  Married-civ-spouse    Protective-serv   
3   44    Private               10  Married-civ-spouse  Machine-op-inspct   
4   18        NaN               10       Never-married                NaN   

  relationship   race  gender  capital-gain  capital-loss  hours-per-week  \
0    Own-child  Black    Male             0             0              40   
1      Husband  White    Male             0             0              50   
2      Husband  White    Male             0             0              40   
3      Husband  Black    Male    

### 2.3. Split features and target
Sau khi tách dữ liệu:
- tập đặc trưng `X` gồm 12 biến,
- biến mục tiêu `y` là `income`.

Phân phối của biến mục tiêu cho thấy số mẫu thuộc lớp `<=50K` lớn hơn đáng kể so với lớp `>50K`, nghĩa là bài toán có hiện tượng mất cân bằng lớp nhẹ.  
Vì vậy, ở bước chia train/test, nhóm sẽ dùng `stratify=y` để giữ phân phối lớp tương đối ổn định giữa hai tập.

In [ ]:
# [Preprocessing 2.3] Split features and target
X = df_prep.drop(columns=['income'])
y = df_prep['income']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

X shape: (48842, 12)
y shape: (48842,)

Target distribution:
income
<=50K    37155
>50K     11687
Name: count, dtype: int64


### 2.4. Identify numerical and categorical features
Các biến được chia thành hai nhóm để phục vụ bước tiền xử lý:

- **Numerical features**: `age`, `educational-num`, `capital-gain`, `capital-loss`, `hours-per-week`
- **Categorical features**: `workclass`, `marital-status`, `occupation`, `relationship`, `race`, `gender`, `native-country`

Việc tách riêng hai nhóm này giúp xây dựng pipeline phù hợp:
- biến số sẽ được xử lý theo các cấu hình preprocessing khác nhau,
- biến phân loại sẽ được xử lý missing value và encoding trước khi đưa vào mô hình.

In [ ]:
# [Preprocessing 2.4] Identify numerical and categorical columns
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

print("Numerical features:", num_features)
print("\nCategorical features:", cat_features)

Numerical features: ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']

Categorical features: ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country']


### 2.5. Train-test split
Dữ liệu được chia theo tỷ lệ **80/20** cho train và test.  
Việc sử dụng `stratify=y` giúp giữ phân phối lớp của biến mục tiêu gần như giống nhau giữa hai tập.

Điều này đặc biệt quan trọng với bài toán phân loại có mất cân bằng lớp, vì giúp việc đánh giá mô hình ở tập test đáng tin cậy hơn.

In [ ]:
# [Preprocessing 2.5] Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).round(4))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).round(4))

X_train shape: (39073, 12)
X_test shape : (9769, 12)

Train target distribution:
income
<=50K    0.7607
>50K     0.2393
Name: proportion, dtype: float64

Test target distribution:
income
<=50K    0.7607
>50K     0.2393
Name: proportion, dtype: float64


### 2.6. Define preprocessing configurations
Để việc tiền xử lý có ý nghĩa hơn, nhóm xây dựng nhiều cấu hình preprocessing khác nhau thay vì chỉ dùng một pipeline cố định.

Các cấu hình được thay đổi theo ba yếu tố chính:
- cách xử lý missing value ở biến categorical,
- cách scaling biến numerical,
- và giữ nguyên One-Hot Encoding cho các biến categorical không có thứ tự.

Lưu ý: các biến numerical trong bộ dữ liệu này không có missing value, nên không cần áp dụng numerical imputation ở bước so sánh này.

In [20]:
# [Preprocessing 2.6] Define multiple preprocessing configurations
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler

preprocessing_configs = {
    "config_1_onehot_mostfreq_standard": ColumnTransformer(transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features)
    ]),

    "config_2_onehot_constant_standard": ColumnTransformer(transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features)
    ]),

    "config_3_onehot_mostfreq_minmax": ColumnTransformer(transformers=[
        ('num', MinMaxScaler(), num_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features)
    ]),

    "config_4_onehot_mostfreq_noscale": ColumnTransformer(transformers=[
        ('num', 'passthrough', num_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_features)
    ])
}

print("Available preprocessing configurations:")
for name in preprocessing_configs.keys():
    print("-", name)

Available preprocessing configurations:
- config_1_onehot_mostfreq_standard
- config_2_onehot_constant_standard
- config_3_onehot_mostfreq_minmax
- config_4_onehot_mostfreq_noscale


### 2.7. Compare preprocessing configurations
Kết quả cho thấy các cấu hình preprocessing tạo ra số chiều đầu ra khác nhau:

- các cấu hình dùng **One-Hot Encoding** làm tăng số lượng đặc trưng,
- cấu hình dùng giá trị hằng `"Missing"` cho categorical missing có thể làm tăng thêm số category sau encoding,
- các lựa chọn scaling không làm thay đổi số chiều nhưng có thể ảnh hưởng đến hiệu năng mô hình ở bước sau.

Điều này cho thấy lựa chọn preprocessing có ảnh hưởng trực tiếp đến cách biểu diễn dữ liệu đầu vào cho mô hình.

In [21]:
# [Preprocessing 2.7] Compare output shape of different preprocessing configurations
processed_results = {}

for name, transformer in preprocessing_configs.items():
    X_train_transformed = transformer.fit_transform(X_train)
    X_test_transformed = transformer.transform(X_test)

    processed_results[name] = {
        'transformer': transformer,
        'X_train_shape': X_train_transformed.shape,
        'X_test_shape': X_test_transformed.shape,
        'train_type': type(X_train_transformed).__name__,
        'test_type': type(X_test_transformed).__name__
    }

for name, result in processed_results.items():
    print(f"\n{name}")
    print("X_train shape:", result['X_train_shape'])
    print("X_test shape :", result['X_test_shape'])
    print("Train type   :", result['train_type'])
    print("Test type    :", result['test_type'])


config_1_onehot_mostfreq_standard
X_train shape: (39073, 88)
X_test shape : (9769, 88)
Train type   : csr_matrix
Test type    : csr_matrix

config_2_onehot_constant_standard
X_train shape: (39073, 91)
X_test shape : (9769, 91)
Train type   : csr_matrix
Test type    : csr_matrix

config_3_onehot_mostfreq_minmax
X_train shape: (39073, 88)
X_test shape : (9769, 88)
Train type   : csr_matrix
Test type    : csr_matrix

config_4_onehot_mostfreq_noscale
X_train shape: (39073, 88)
X_test shape : (9769, 88)
Train type   : csr_matrix
Test type    : csr_matrix


### 2.8. Summarize preprocessing configurations
Bảng dưới đây tổng hợp số chiều đầu ra và kiểu dữ liệu của từng cấu hình preprocessing để thuận tiện cho việc so sánh.

In [23]:
# [Preprocessing 2.8] Summarize preprocessing configurations in a table
comparison_rows = []

for name, result in processed_results.items():
    comparison_rows.append({
        'configuration': name,
        'X_train_shape': result['X_train_shape'],
        'X_test_shape': result['X_test_shape'],
        'train_type': result['train_type'],
        'test_type': result['test_type'],
        'n_features_after_preprocessing': result['X_train_shape'][1]
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df = comparison_df.sort_values(
    by='n_features_after_preprocessing',
    ascending=False
).reset_index(drop=True)

print(comparison_df)

                       configuration X_train_shape X_test_shape  train_type  \
0  config_2_onehot_constant_standard   (39073, 91)   (9769, 91)  csr_matrix   
1  config_1_onehot_mostfreq_standard   (39073, 88)   (9769, 88)  csr_matrix   
2    config_3_onehot_mostfreq_minmax   (39073, 88)   (9769, 88)  csr_matrix   
3   config_4_onehot_mostfreq_noscale   (39073, 88)   (9769, 88)  csr_matrix   

    test_type  n_features_after_preprocessing  
0  csr_matrix                              91  
1  csr_matrix                              88  
2  csr_matrix                              88  
3  csr_matrix                              88  


### 2.9. Evaluate preprocessing configurations with Logistic Regression
Để chọn cấu hình preprocessing phù hợp, nhóm sử dụng Logistic Regression như một mô hình đánh giá cố định và so sánh hiệu năng của các cấu hình theo các chỉ số classification.

In [25]:
# [Preprocessing 2.9] Compare preprocessing configurations using Logistic Regression
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings("ignore", category=ConvergenceWarning)

lr_results = []

for name, transformer in preprocessing_configs.items():
    X_train_transformed = transformer.fit_transform(X_train)
    X_test_transformed = transformer.transform(X_test)

    model = LogisticRegression(
        random_state=42,
        max_iter=5000,
        solver='lbfgs'
    )
    model.fit(X_train_transformed, y_train)
    y_pred = model.predict(X_test_transformed)

    lr_results.append({
        'configuration': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, pos_label='>50K'),
        'recall': recall_score(y_test, y_pred, pos_label='>50K'),
        'f1_score': f1_score(y_test, y_pred, pos_label='>50K')
    })

lr_results_df = pd.DataFrame(lr_results) \
    .sort_values(by='f1_score', ascending=False) \
    .reset_index(drop=True)

print(lr_results_df)

                       configuration  accuracy  precision    recall  f1_score
0  config_2_onehot_constant_standard  0.855359   0.746142  0.599658  0.664928
1   config_4_onehot_mostfreq_noscale  0.853107   0.740287  0.594953  0.659711
2  config_1_onehot_mostfreq_standard  0.852185   0.737513  0.593670  0.657820
3    config_3_onehot_mostfreq_minmax  0.852595   0.744022  0.585543  0.655337


### 2.10. Select best preprocessing configuration
Dựa trên kết quả so sánh với Logistic Regression, nhóm chọn cấu hình preprocessing tốt nhất để sử dụng cho các bước tiếp theo.

In [26]:
# [Preprocessing 2.10] Select best preprocessing configuration
best_preprocessor = preprocessing_configs['config_2_onehot_constant_standard']

X_train_best = best_preprocessor.fit_transform(X_train)
X_test_best = best_preprocessor.transform(X_test)

print("Best config selected: config_2_onehot_constant_standard")
print("X_train_best shape:", X_train_best.shape)
print("X_test_best shape :", X_test_best.shape)

Best config selected: config_2_onehot_constant_standard
X_train_best shape: (39073, 91)
X_test_best shape : (9769, 91)


Dựa trên kết quả so sánh bằng Logistic Regression như một mô hình baseline cố định, nhóm chọn `config_2_onehot_constant_standard` làm cấu hình preprocessing được sử dụng cho các bước thử nghiệm tiếp theo, vì cấu hình này đạt F1-score cao nhất trong các cấu hình preprocessing đã thử.

### Preprocessing Summary
Sau bước tiền xử lý, nhóm đã:
- Thay thế `?` bằng `NaN` để chuẩn hóa missing value
- Loại bỏ `education` (trùng với `educational-num`) và `fnlwgt` (sampling weight)
- Chia train/test theo tỷ lệ 80/20 với `stratify=y`
- Thử nghiệm 4 cấu hình preprocessing và chọn `config_2_onehot_constant_standard` dựa trên F1-score

Dữ liệu sau preprocessing có **91 đặc trưng**, sẵn sàng cho bước huấn luyện mô hình.